## GeoProxy · 01 — Earth terrain features → `sites.json`

**What this notebook does:** for each Earth site in the list below, it cuts a square window of elevation data around the site, puts it on a common 500 m grid, measures three terrain features (relief, slope, roughness) and saves everything as `sites.json` for the web app.

**How to run:** `Runtime → Run all`. The first run takes about 1–2 minutes.

**To add a site:** add an entry to `SITES` in the configuration cell, then run all again. Elevation tiles are fetched automatically from the public Copernicus DEM archive, so any land location on Earth works.

Output: `sites.json`, saved to your Drive (`geoproxy-data/processed/`) when Drive is mounted, and also downloaded to your computer.

### 1 · Install the GeoTIFF reader
*Why:* `rasterio` reads geographic raster files (GeoTIFF) such as elevation models. Colab doesn't always include it.

In [ ]:
!pip -q install rasterio

### 2 · Configuration — the only cell you normally edit
*Why:* everything you may want to change (which sites, window size, resolution) is in one place, so the rest of the notebook never needs editing.

- `lat` / `lon` = centre of the analysis window, in decimal degrees (south and west are negative).
- `analog_for` and `notes` record **your team's hypothesis** about why the site is interesting. They are not measurements.
- `PIXEL_M = 500` is chosen to match the coarsest planetary dataset we plan to use (Mars MOLA MEGDR, about 463 m per pixel). **Every body must be measured on the same grid**, otherwise slope and roughness are not comparable.

In [ ]:
import os

SITES = [
    {
        "id": "mount-cameroon",
        "name": "Mount Cameroon",
        "region": "Southwest Region, Cameroon",
        "lat": 4.217, "lon": 9.167,          # summit area
        "analog_for": ["mars"],
        "notes": "Active basaltic volcano. Team hypothesis: analog for Martian volcanic terrain.",
    },
    # Add more sites like this (uncomment and edit):
    # {"id": "atacama-yungay", "name": "Yungay, Atacama Desert", "region": "Antofagasta, Chile",
    #  "lat": -24.08, "lon": -70.0, "analog_for": ["mars"], "notes": "Hyper-arid desert."},
]

WINDOW_KM = 20      # side of the square analysis window (km)
PIXEL_M = 500       # common grid spacing for ALL bodies (m)
DRIVE_DATA = "/content/drive/MyDrive/geoproxy/geoproxy-data"   # your team's Drive data folder

# Read remote GeoTIFFs efficiently (skip directory listings over HTTP)
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".tif"

### 3 · Connect Google Drive (optional)
*Why:* lets the notebook use elevation tiles already in your Drive and save results there. Colab asks you to allow access. If you skip it, results go to `/content/output` and are still downloaded.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive not mounted:", e)

OUT_DIR = os.path.join(DRIVE_DATA, "processed") if os.path.isdir(DRIVE_DATA) else "/content/output"
os.makedirs(OUT_DIR, exist_ok=True)
print("Results will be saved to:", OUT_DIR)

### 4 · Find the elevation tiles for a site
*Why:* Copernicus DEM GLO-90 (ESA, about 90 m per pixel) is split into 1°×1° tiles named after their south-west corner. These helpers build a tile's name from a latitude/longitude, then use your Drive copy when it exists or else stream it from the public archive on AWS.

In [ ]:
import math
import numpy as np
import rasterio
from rasterio.merge import merge
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
from rasterio.crs import CRS

EARTH_RADIUS_M = 6371000

def tile_name(lat_floor, lon_floor):
    ns = "N" if lat_floor >= 0 else "S"
    ew = "E" if lon_floor >= 0 else "W"
    return f"Copernicus_DSM_COG_30_{ns}{abs(lat_floor):02d}_00_{ew}{abs(lon_floor):03d}_00_DEM"

def tile_path(lat_floor, lon_floor):
    name = tile_name(lat_floor, lon_floor)
    local = os.path.join(DRIVE_DATA, "raw", "earth", name + ".tif")
    if os.path.exists(local):
        return local
    return f"/vsicurl/https://copernicus-dem-90m.s3.amazonaws.com/{name}/{name}.tif"

print(tile_name(4, 9), "->", tile_path(4, 9))

### 5 · Cut a window and put it on the common grid
*Why:* raw tiles are in degrees, and a degree of longitude shrinks towards the poles. We reproject each window onto a flat metric map centred on the site (azimuthal equidistant projection), at exactly `PIXEL_M` metres per pixel, averaging the 90 m pixels into each 500 m pixel. The same procedure will be used for the Moon and Mars, which keeps the comparison fair.

In [ ]:
def bounds_deg(lat, lon, half_m, radius_m):
    dlat = math.degrees(half_m / radius_m)
    dlon = dlat / max(math.cos(math.radians(lat)), 0.01)
    return lon - dlon, lat - dlat, lon + dlon, lat + dlat

def earth_window(lat, lon):
    """Elevation (m) on a square grid of WINDOW_KM x WINDOW_KM at PIXEL_M spacing, centred on (lat, lon)."""
    half = WINDOW_KM * 1000 / 2
    west, south, east, north = bounds_deg(lat, lon, half * 1.5, EARTH_RADIUS_M)  # 1.5x margin for the reprojection
    paths = [tile_path(a, b)
             for a in range(math.floor(south), math.floor(north) + 1)
             for b in range(math.floor(west), math.floor(east) + 1)]
    sources = []
    for p in paths:
        try:
            sources.append(rasterio.open(p))
        except Exception:
            print("   tile not available (open ocean or withheld):", p)
    if not sources:
        raise RuntimeError("No elevation tiles found for this location")

    mosaic, src_transform = merge(sources, bounds=(west, south, east, north), nodata=-9999)
    for s in sources:
        s.close()
    src = mosaic[0].astype("float64")
    src[src == -9999] = np.nan

    n = int(round(WINDOW_KM * 1000 / PIXEL_M)) + 1          # odd size, so the site is the centre pixel
    dst = np.full((n, n), np.nan, dtype="float64")
    dst_crs = CRS.from_proj4(f"+proj=aeqd +lat_0={lat} +lon_0={lon} +datum=WGS84 +units=m")
    dst_transform = from_origin(-n * PIXEL_M / 2, n * PIXEL_M / 2, PIXEL_M, PIXEL_M)
    reproject(source=src, destination=dst,
              src_transform=src_transform, src_crs="EPSG:4326", src_nodata=np.nan,
              dst_transform=dst_transform, dst_crs=dst_crs, dst_nodata=np.nan,
              resampling=Resampling.average)
    return dst

### 6 · Measure the terrain features
*Why:* we use only features that mean the same thing on every body. **Absolute elevation is deliberately excluded**, because "zero" is sea level on Earth, the areoid on Mars and a reference sphere on the Moon.

| Feature | Definition |
|---|---|
| `relief_m` | 90th minus 10th percentile of elevation in the window. Robust height range, ignoring single spikes |
| `slope_deg` | Median slope of the window, from the gradient of the 500 m grid |
| `roughness_m` | Mean absolute height difference between each cell and its 8 neighbours (Terrain Ruggedness Index, Wilson et al. 2007) |

A window is rejected (features set to `null`) if less than 95% of it has valid data.

In [ ]:
MIN_COVERAGE = 0.95

def terrain_features(z):
    coverage = float(np.isfinite(z).mean())
    if coverage < MIN_COVERAGE:
        return None, coverage
    gy, gx = np.gradient(z, PIXEL_M)
    slope = np.degrees(np.arctan(np.hypot(gx, gy)))
    c = z[1:-1, 1:-1]
    rows, cols = z.shape
    diffs = [np.abs(z[1 + dy:rows - 1 + dy, 1 + dx:cols - 1 + dx] - c)
             for dy in (-1, 0, 1) for dx in (-1, 0, 1) if (dy, dx) != (0, 0)]
    tri = np.mean(diffs, axis=0)
    return {
        "relief_m": float(np.nanpercentile(z, 90) - np.nanpercentile(z, 10)),
        "slope_deg": float(np.nanmedian(slope)),
        "roughness_m": float(np.nanmean(tri)),
    }, coverage

### 7 · Run it for every site and look at the result
*Why:* always look at the data, not just the numbers. A quick map of each window catches mistakes such as a wrong coordinate or a window that falls in the sea.

In [ ]:
import matplotlib.pyplot as plt

measured = []
for site in SITES:
    print(f"Measuring {site['name']} ({site['lat']}, {site['lon']}) ...")
    z = earth_window(site["lat"], site["lon"])
    feats, coverage = terrain_features(z)
    valid = z[np.isfinite(z)]
    sea_fraction = float(np.mean(valid <= 0)) if valid.size else None
    measured.append({"site": site, "features": feats, "coverage": coverage, "sea_fraction": sea_fraction})
    print("   features:", {k: round(v, 2) for k, v in feats.items()} if feats else None,
          "| coverage:", round(coverage, 3), "| sea fraction:", sea_fraction)

    extent_km = WINDOW_KM / 2
    plt.figure(figsize=(5, 4))
    plt.imshow(z, cmap="terrain", extent=[-extent_km, extent_km, -extent_km, extent_km])
    plt.colorbar(label="elevation (m)")
    plt.title(f"{site['name']} · {WINDOW_KM} km window · {PIXEL_M} m grid")
    plt.xlabel("km east of site"); plt.ylabel("km north of site")
    plt.show()

### 8 · Build `sites.json` (the data contract)
*Why:* the web app relies on this exact structure. Every feature carries its value, unit, **how it was obtained** (`measured`, `literature` or `missing`) and its source, so the app can show users where each number comes from. Earth sites and Moon/Mars targets live in separate files with the same feature names.

In [ ]:
import json
from datetime import datetime, timezone

UNITS = {"relief_m": "m", "slope_deg": "degrees", "roughness_m": "m"}

def feature_entry(value, unit):
    return {
        "value": None if value is None else round(value, 2),
        "unit": unit,
        "method": "measured" if value is not None else "missing",
        "source": "copernicus-glo90",
    }

sites_json = {
    "schema_version": "0.1",
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "method": {
        "window_km": WINDOW_KM,
        "pixel_m": PIXEL_M,
        "projection": "azimuthal equidistant, centred on each site",
        "resampling": "average",
        "min_coverage": MIN_COVERAGE,
        "features": {
            "relief_m": "90th minus 10th percentile of elevation in the window (m)",
            "slope_deg": "median slope of the window, from 500 m grid gradients (degrees)",
            "roughness_m": "mean absolute elevation difference between each cell and its 8 neighbours (TRI, Wilson et al. 2007) (m)",
        },
    },
    "sources": {
        "copernicus-glo90": {
            "dataset": "Copernicus DEM GLO-90 (digital surface model, about 90 m)",
            "provider": "ESA / European Union, Copernicus programme; accessed via AWS Open Data",
            "url": "https://registry.opendata.aws/copernicus-dem/",
            "attribution": "© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA",
            "caveat": "Surface model: includes vegetation canopy and buildings, mostly smoothed out at 500 m.",
        }
    },
    "sites": [],
}

for m in measured:
    s, f = m["site"], m["features"] or {}
    sites_json["sites"].append({
        "id": s["id"],
        "body": "earth",
        "name": s["name"],
        "region": s["region"],
        "lat": s["lat"],
        "lon": s["lon"],
        "analog_for": s["analog_for"],
        "notes": s["notes"],
        "features": {k: feature_entry(f.get(k), u) for k, u in UNITS.items()},
        "qc": {"coverage": round(m["coverage"], 3),
               "sea_fraction": None if m["sea_fraction"] is None else round(m["sea_fraction"], 3)},
    })

print(json.dumps(sites_json, indent=2))

### 9 · Save and download
*Why:* saving to Drive keeps the team's copy up to date. The download gives you a file to put in the repo at `data/sites.json`.

In [ ]:
out_path = os.path.join(OUT_DIR, "sites.json")
with open(out_path, "w", encoding="utf-8") as fh:
    json.dump(sites_json, fh, indent=2, ensure_ascii=False)
print("Saved:", out_path)

try:
    from google.colab import files
    files.download(out_path)
except Exception as e:
    print("Download skipped:", e)